In [ ]:
''' Functions in deblur flow '''

import numpy as np
import cv2 as cv
from scipy import ndimage, signal
from scipy.signal import convolve2d
import scipy.signal

import sys
DBL_MIN = sys.float_info.min
import matplotlib.pyplot as plt

try:
    import cupy as cp
    import cupyx.scipy.signal as cpx_signal
except ImportError:
    cp = None
    cpx_signal = None


### Common functions

In [ ]:
def img_conversion(img_in, to_float):
  """ img range conversion
          Args:
              img_in (ndarray, shape(ch, height, width))
                  dtype = uint8 (range: 0-255) or float (range: 0-1)
              to_float (bool): 1 means from uint8 to float; 0 means from float to uint8
          Returns:
              img_out (ndarray, shape(ch, height, width))
                  dtype = float (range: 0-1) or uint8 (range: 0-255)
  """

  ''' TODO '''

  if to_float:

    if img_in.dtype == np.uint8:

      img_out = (img_in / 255.0).astype(np.float64)

    else:

      return img_in

  else:
    if img_in.dtype == np.float64:

      img_out = (np.round(np.clip(img_in, 0.0, 1.0) * 255)).astype(np.uint8)

    else:

      return img_in

  return img_out

In [ ]:

def kernel_conversion (k_in):
  """ kernal preprocess
          Args:
              k_in (uint8 ndarray, shape(height, width)): Blur kernel
          Returns:
              k_out (float ndarray, shape(height, width)): Blur kernel
  """

  ''' TODO '''

  k_out = (k_in / np.sum(k_in)).astype(np.float64)

  return k_out

### Wiener Deconvolution

In [ ]:
def kernel_preprocess(k_in, img_height, img_width):
  """ kernel_preprocess
          Args:
              k_in (uint8 ndarray, shape(height, width)): Blur kernel
              img_height (int): photo height
              img_width (int): photo width

          Returns:
              k_pad (uint8 ndarray, shape(img_height, img_width)): Blur kernel after preprocessing

          Todo:
              kernel preprocess for Wiener deconvolution
  """

  ''' TODO '''

  kh, kw = k_in.shape

  k_pad = np.zeros((img_height, img_width), dtype = 'uint8')
  k_pad[0:kh, 0:kw] = k_in

  cy = kh // 2
  cx = kw // 2

  k_pad = np.roll(k_pad, shift = (-cy, -cx), axis = (0, 1))

  return k_pad


In [ ]:
def Wiener(img_in, k_in, SNR_F):
  """ Wiener deconvolution
          Args:
              img_in (uint8 ndarray, shape(height, width, ch)): Blurred image
              k_in (uint8 ndarray, shape(height, width)): Blur kernel
              SNR_F (float): Wiener deconvolution parameter
          Returns:
              Wiener_result (uint8 ndarray, shape(height, width, ch)): Wiener-deconv image

          To do:
              Wiener deconvolution (Note that you shall be calling img_conversion, kernel_conversion and kernel_preprocess)
  """

  ''' TODO '''
  b = img_conversion(img_in, True)
  img_h, img_w, ch = b.shape
  k = kernel_conversion(kernel_preprocess(k_in, img_h, img_w))
  K = np.fft.rfft2(k)

  Wiener_result = np.empty(b.shape, dtype = 'uint8')

  for c in range(ch):

    B = np.fft.rfft2(b[:, :, c])

    I = B * (np.conj(K) / (np.abs(K) ** 2 + (1 / SNR_F)))
    I_hat = np.fft.irfft2(I)

    Wiener_result[:, :, c] = img_conversion(I_hat, False)

  return Wiener_result

In [ ]:
# Optimized Wiener algorithm (cuda)
def Wiener_cuda(img_in, k_in, SNR_F):
  """ Wiener deconvolution
          Args:
              img_in (uint8 ndarray, shape(height, width, ch)): Blurred image
              k_in (uint8 ndarray, shape(height, width)): Blur kernel
              SNR_F (float): Wiener deconvolution parameter
          Returns:
              Wiener_result (uint8 ndarray, shape(height, width, ch)): Wiener-deconv image

          To do:
              Wiener deconvolution (Note that you shall be calling img_conversion, kernel_conversion and kernel_preprocess)
  """

  ''' TODO '''
  if cp is None:
    raise ImportError('cupy is required for Wiener_cuda.')
  b = cp.asarray(img_conversion(img_in, True))
  img_h, img_w, ch = b.shape
  k = cp.asarray(kernel_conversion(kernel_preprocess(k_in, img_h, img_w)))[:, :, cp.newaxis]
  K = cp.fft.rfft2(k, axes = (0, 1))

  Wiener_filter = cp.conj(K) / (cp.abs(K) ** 2 + (1 / SNR_F))
  B = cp.fft.rfft2(b, axes = (0, 1))

  I = B * Wiener_filter
  I_hat = cp.fft.irfft2(I, axes = (0, 1))

  Wiener_result = img_conversion(cp.asnumpy(I_hat), False)

  return Wiener_result

### RL Deconvolution

In [ ]:
def check_RL_energy(img_in, k_in, I_in):
  """ Check RL Energy
          Args:
              img_in (uint8 ndarray, shape(height, width, ch)): Blurred image
              k_in (uint8 ndarray, shape(height, width)): Blur kernel
              I_in (uint8 ndarray, shape(height, width, ch)): Deblurred image

          Returns:
              RL_energy (float): RL_energy

          Todo:
              Calculate RL energy (Note that you shall be calling img_conversion and kernel_conversion)
  """

  ''' TODO '''

  b = img_conversion(img_in, True)
  k = kernel_conversion(k_in)
  i = img_conversion(I_in, True)

  RL_energy = 0.0

  for c in range(b.shape[2]):

    B = b[:, :, c]
    I = i[:, :, c]
    I_conv_K = scipy.signal.convolve2d(I, k, mode = "same", boundary = 'symm')
    RL_energy += np.sum(I_conv_K - B * np.log(I_conv_K + DBL_MIN))

  return RL_energy

In [ ]:
def RL(img_in, k_in, max_iter):
  """ RL deconvolution
          Args:
              img_in (uint8 ndarray, shape(height, width, ch)): Blurred image
              k_in (uint8 ndarray, shape(height, width)): blur kernel
              max_iter (int): total iteration count

          Returns:
              RL_result (uint8 ndarray, shape(height, width, ch)): RL-deblurred image

          Todo:
              RL deconvolution (Note that you shall be calling img_conversion and kernel_conversion)
  """

  ''' TODO '''

  b = img_conversion(img_in, True)
  k = kernel_conversion(k_in)
  k_adj = k[::-1, ::-1]

  I = b.copy()

  for i in range(max_iter):

    for c in range(b.shape[2]):

      I[:, :, c] = I[:, :, c] * scipy.signal.convolve2d(b[:, :, c] / (scipy.signal.convolve2d(I[:, :, c], k, mode = 'same', boundary = 'symm') + DBL_MIN), k_adj, mode = 'same', boundary = 'symm')

  RL_result = img_conversion(I, False)

  return RL_result

In [ ]:
# Optimized RL algorithm (cuda)
def RL_cuda(img_in, k_in, max_iter):
  """ RL deconvolution
          Args:
              img_in (uint8 ndarray, shape(height, width, ch)): Blurred image
              k_in (uint8 ndarray, shape(height, width)): blur kernel
              max_iter (int): total iteration count

          Returns:
              RL_result (uint8 ndarray, shape(height, width, ch)): RL-deblurred image

          Todo:
              RL deconvolution (Note that you shall be calling img_conversion and kernel_conversion)
  """

  ''' TODO '''
  if cp is None or cpx_signal is None:
    raise ImportError('cupy is required for RL_cuda.')

  b = cp.asarray(img_conversion(img_in, True))
  k = cp.asarray(kernel_conversion(k_in))[:, :, cp.newaxis]
  k_adj = k[::-1, ::-1, :]

  I = b.copy()

  pad_h = k.shape[0] // 2
  pad_w = k.shape[1] // 2

  for i in range(max_iter):

    I_pad = cp.pad(I, pad_width = ((pad_h, pad_h), (pad_w, pad_w), (0, 0)), mode = 'symmetric')
    pred_B = cpx_signal.fftconvolve(I_pad, k, mode = 'valid', axes = (0, 1))
    pred_B = cp.maximum(pred_B, DBL_MIN)

    ratio = b / pred_B
    ratio_pad = cp.pad(ratio, pad_width = ((pad_h, pad_h), (pad_w, pad_w), (0, 0)), mode = 'symmetric')
    error = cpx_signal.fftconvolve(ratio_pad, k_adj, mode = 'valid', axes = (0, 1))

    I *= error

  RL_result = img_conversion(cp.asnumpy(I), False)

  return RL_result

### BRL Deconvolution

In [ ]:
def BRL_EB(I_in, sigma_r, rk):
  """ BRL Edge-preserving regularization term
          Args:
              I_in (uint8 ndarray, shape(ch, height, width)): Deblurred image
              sigma_r (float): BRL parameter
              rk (int): BRL parameter

          Returns:
              EB (float ndarray, shape(ch)): Edge-preserving regularization term

          Todo:
              Calculate BRL Edge-preserving regularization term
  """

  ''' TODO '''

  I_x = img_conversion(I_in, True)
  h, w, ch = I_x.shape

  r_omega = rk // 2
  sigma_s = (r_omega / 3.0) ** 2

  EB = np.zeros(ch, dtype = 'float64')
  I_pad = np.pad(I_x, pad_width = ((r_omega, r_omega), (r_omega, r_omega), (0, 0)), mode = 'symmetric')

  for dy in range(-r_omega, r_omega + 1):
    for dx in range(-r_omega, r_omega + 1):

      f = np.exp(-(dy ** 2 + dx ** 2) / (2 * sigma_s))

      I_y = I_pad[r_omega + dy : r_omega + dy + h, r_omega + dx : r_omega + dx + w, :]
      rho = 1 - np.exp(-((I_x - I_y) ** 2) / (2 * sigma_r))

      EB += np.sum(f * rho, axis = (0, 1))

  return EB

In [ ]:
def BRL_energy(img_in, k_in, I_in, lamb_da, sigma_r, rk):
    """ BRL Energy
            Args:
                img_in (uint8 ndarray, shape(height, width, ch)): Blurred image
                k_in (uint8 ndarray, shape(height, width)): Blur kernel
                I_in (uint8 ndarray, shape(height, width, ch)): Deblurred image
                lamb_da (float): BRL parameter
                sigma_r (float): BRL parameter
                rk (int): BRL parameter

            Returns:
                BRL_energy (float): BRL_energy

            Todo:
                Calculate BRL energy (Note that you shall be calling img_conversion, kernel_conversion and BRL_EB)
    """

    ''' TODO '''

    BRL_energy = check_RL_energy(img_in, k_in, I_in) + lamb_da * np.sum(BRL_EB(I_in, sigma_r, rk))

    return BRL_energy

In [ ]:
def BRL(img_in, k_in, max_iter, lamb_da, sigma_r, rk):
    """ BRL deconvolution
            Args:
                img_in (uint8 ndarray, shape(height, width, ch)): Blurred image
                k_in (uint8 ndarray, shape(height, width)): Blur kernel
                max_iter (int): Total iteration count
                lamb_da (float): BRL parameter
                sigma_r (float): BRL parameter
                rk (int): BRL parameter

            Returns:
                BRL_result (uint8 ndarray, shape(height, width, ch)): BRL-deblurred image

            Todo:
                BRL deconvolution (Note that you shall be calling img_conversion and kernel_conversion)
    """

    ''' TODO '''

    b = img_conversion(img_in, True)
    h, w, ch = b.shape
    k = kernel_conversion(k_in)
    k_adj = k[::-1, ::-1]

    I_x = b.copy()

    r_omega = rk // 2
    sigma_s = (r_omega / 3.0) ** 2

    for i in range(max_iter):

      I_pad = np.pad(I_x, pad_width = ((r_omega, r_omega), (r_omega, r_omega), (0, 0)), mode = 'symmetric')
      grad_EB = np.zeros_like(I_x)

      for dy in range(-r_omega, r_omega + 1):
        for dx in range(-r_omega, r_omega + 1):

          f = np.exp(-(dy ** 2 + dx ** 2) / (2 * sigma_s))

          I_y = I_pad[r_omega + dy : r_omega + dy + h, r_omega + dx : r_omega + dx + w, :]
          g = np.exp(-((I_x - I_y) ** 2) / (2 * sigma_r))

          grad_EB += 2 * f * g * (I_x - I_y) / sigma_r

      for c in range(ch):

        I_x[:, :, c] = I_x[:, :, c] * scipy.signal.convolve2d(b[:, :, c] / (scipy.signal.convolve2d(I_x[:, :, c], k, mode = 'same', boundary = 'symm') + DBL_MIN), k_adj, mode = 'same', boundary = 'symm')
        I_x[:, :, c] /= 1 + lamb_da * grad_EB[:, :, c]

    BRL_result = img_conversion(I_x, False)

    return BRL_result

In [ ]:
# Optimized BRL algorithm (fft)
def BRL_fft(img_in, k_in, max_iter, lamb_da, sigma_r, rk):
  """ BRL deconvolution
          Args:
              img_in (uint8 ndarray, shape(height, width, ch)): Blurred image
              k_in (uint8 ndarray, shape(height, width)): Blur kernel
              max_iter (int): Total iteration count
              lamb_da (float): BRL parameter
              sigma_r (float): BRL parameter
              rk (int): BRL parameter

          Returns:
              BRL_result (uint8 ndarray, shape(height, width, ch)): BRL-deblurred image

          Todo:
              BRL deconvolution (Note that you shall be calling img_conversion and kernel_conversion)
  """

  ''' TODO '''

  b = img_conversion(img_in, True)
  h, w, ch = b.shape
  k = kernel_conversion(k_in)[:, :, np.newaxis]
  k_adj = k[::-1, ::-1]

  I_x = b.copy()

  r_omega = rk // 2
  sigma_s = (r_omega / 3.0) ** 2

  pad_h = k.shape[0] // 2
  pad_w = k.shape[1] // 2

  for i in range(max_iter):

    I_pad = np.pad(I_x, pad_width = ((r_omega, r_omega), (r_omega, r_omega), (0, 0)), mode = 'symmetric')
    grad_EB = np.zeros_like(I_x)

    for dy in range(-r_omega, r_omega + 1):
      for dx in range(-r_omega, r_omega + 1):

        f = np.exp(-(dy ** 2 + dx ** 2) / (2 * sigma_s))

        I_y = I_pad[r_omega + dy : r_omega + dy + h, r_omega + dx : r_omega + dx + w, :]
        g = np.exp(-((I_x - I_y) ** 2) / (2 * sigma_r))

        grad_EB += f * g * (I_x - I_y)

    grad_EB *= 2 / sigma_r

    I_x_pad = np.pad(I_x, pad_width = ((pad_h, pad_h), (pad_w, pad_w), (0, 0)), mode = 'symmetric')
    pred_B = scipy.signal.fftconvolve(I_x_pad, k, mode = 'valid', axes = (0, 1)) # I ⊗ K
    pred_B = np.maximum(pred_B, DBL_MIN)
    ratio = np.pad(b / (pred_B + DBL_MIN), pad_width = ((pad_h, pad_h), (pad_w, pad_w), (0, 0)), mode = 'symmetric') # B / (I ⊗ K)
    error = scipy.signal.fftconvolve(ratio, k_adj, mode = 'valid', axes = (0, 1)) # K* ⊗ (B / (I ⊗ K))

    I_x = (I_x / (1 + lamb_da * grad_EB)) * error

  BRL_result = img_conversion(I_x, False)

  return BRL_result

In [ ]:
# Optimized BRL algorithm (fft + jit + cuda)
import math
from numba import cuda

@cuda.jit
def compute_grad_EB_cuda_kernel(I_x, I_pad, grad_EB, r_omega, sigma_s, sigma_r):

  y, x, c = cuda.grid(3)
  h, w, ch = I_x.shape

  if y < h and x < w and c < ch:

    val = 0.0
    I_x_val = I_x[y, x ,c]

    for dy in range(-r_omega, r_omega + 1):
      for dx in range(-r_omega, r_omega + 1):

        f = math.exp(-(dy * dy + dx * dx) / (2 * sigma_s))
        I_y_val = I_pad[r_omega + dy + y, r_omega + dx + x, c]
        diff = I_x_val - I_y_val
        g = math.exp(-(diff * diff) / (2 * sigma_r))
        val += f * g * diff

    grad_EB[y, x, c] = val

def BRL_cuda_jit(img_in, k_in, max_iter, lamb_da, sigma_r, rk):
  """ BRL deconvolution
          Args:
              img_in (uint8 ndarray, shape(height, width, ch)): Blurred image
              k_in (uint8 ndarray, shape(height, width)): Blur kernel
              max_iter (int): Total iteration count
              lamb_da (float): BRL parameter
              sigma_r (float): BRL parameter
              rk (int): BRL parameter

          Returns:
              BRL_result (uint8 ndarray, shape(height, width, ch)): BRL-deblurred image

          Todo:
              BRL deconvolution (Note that you shall be calling img_conversion and kernel_conversion)
  """

  ''' TODO '''
  if cp is None or cpx_signal is None:
    raise ImportError('cupy is required for BRL_cuda_jit.')

  b = cp.asarray(img_conversion(img_in, True))
  h, w, ch = b.shape
  k = cp.asarray(kernel_conversion(k_in)[:, :, np.newaxis])
  k_adj = k[::-1, ::-1]

  I_x = b.copy()

  r_omega = rk // 2
  sigma_s = (r_omega / 3.0) ** 2

  pad_h = k.shape[0] // 2
  pad_w = k.shape[1] // 2

  threads_per_block = (8, 8, 3)
  blocks_per_grid_y = math.ceil(h / threads_per_block[0])
  blocks_per_grid_x = math.ceil(w / threads_per_block[1])
  blocks_per_grid_c = math.ceil(ch / threads_per_block[2])
  blocks_per_grid = (blocks_per_grid_y, blocks_per_grid_x, blocks_per_grid_c)

  for i in range(max_iter):

    I_pad = cp.pad(I_x, pad_width = ((r_omega, r_omega), (r_omega, r_omega), (0, 0)), mode = 'symmetric')
    grad_EB = cp.zeros_like(I_x)

    compute_grad_EB_cuda_kernel[blocks_per_grid, threads_per_block](I_x, I_pad, grad_EB, r_omega, sigma_s, sigma_r)
    grad_EB *= 2 / sigma_r


    I_x_pad = cp.pad(I_x, pad_width = ((pad_h, pad_h), (pad_w, pad_w), (0, 0)), mode = 'symmetric')
    pred_B = cpx_signal.fftconvolve(I_x_pad, k, mode = 'valid', axes = (0, 1)) # I ⊗ K
    pred_B = cp.maximum(pred_B, DBL_MIN)
    ratio = cp.pad(b / (pred_B + DBL_MIN), pad_width = ((pad_h, pad_h), (pad_w, pad_w), (0, 0)), mode = 'symmetric') # B / (I ⊗ K)
    error = cpx_signal.fftconvolve(ratio, k_adj, mode = 'valid', axes = (0, 1)) # K* ⊗ (B / (I ⊗ K))

    I_x = (I_x / (1 + lamb_da * grad_EB)) * error

  BRL_result = img_conversion(cp.asnumpy(I_x), False)

  return BRL_result